# Stage 1 — Graph Attention Network on Edge-IIoTset

**Question this notebook answers:** does seeing a flow *in the context of the other
flows around it* detect attacks that a per-flow model misses?

---

### The design, and why

| Choice | Reason |
|---|---|
| Each **flow** is a node | The task is per-flow classification, so flows are the things being labelled |
| Edges link flows that **share an endpoint and are close in time** | That is what 'context' means here |
| IP addresses build the edges but are **never a feature** | Otherwise the model just memorises the attacker's address — useless on any other network |
| An **MLP on the same features** is trained alongside | Without this control there is no way to claim the graph helped |
| Evaluation held out **whole attack types** | The autoencoder never saw an attack; a supervised model that had seen them all would win unfairly |

> The MLP control is the heart of the experiment. Same features, same labels, same
> split — the only difference is the graph. Any gap between them is the graph's doing.


## 0 — Configuration


In [ ]:
CSV_PATH = r'C:\Users\KimoStore\Desktop\PoC\DNN-EdgeIIoT-dataset.csv'

# Flows are sampled in CONTIGUOUS blocks, never at random: a burst of attack
# traffic only looks like a burst if its rows stay next to each other.
SAMPLE_ROWS   = 300_000
BLOCK         = 20_000     # size of each contiguous block taken

GRAPH_WINDOW  = 512        # flows are only linked within a block of this many
MAX_DEGREE    = 8          # neighbour cap, keeps edge count linear

# Attack families held out of training entirely — the zero-day test set
HELDOUT_ATTACKS = ['XSS', 'SQL_injection', 'Backdoor']

EPOCHS, LR, HIDDEN, HEADS = 150, 0.01, 32, 4
SEED = 42
OUT_DIR = 'GNN_results'

import os, gc, time, json
import numpy as np, pandas as pd, torch
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
torch.set_num_threads(os.cpu_count())
print('torch', torch.__version__, '| threads', torch.get_num_threads())
assert os.path.exists(CSV_PATH), f'CSV not found: {CSV_PATH}'


## 1 — Load flows, keeping the endpoints this time

The autoencoder dropped `ip.src_host` and `ip.dst_host`. Here they are loaded —
but only to decide which flows are linked. They are removed before the features
are assembled.


In [ ]:
peek = pd.read_csv(CSV_PATH, nrows=2000, low_memory=False)
LABEL_COL, TYPE_COL = 'Attack_label', 'Attack_type'
SRC_COL,  DST_COL   = 'ip.src_host', 'ip.dst_host'
for col in (LABEL_COL, TYPE_COL, SRC_COL, DST_COL):
    assert col in peek.columns, f'missing column: {col}'

ID_LIKE = {'frame.time','ip.src_host','ip.dst_host','arp.src.proto_ipv4',
           'arp.dst.proto_ipv4','http.file_data','http.request.full_uri',
           'http.request.version','icmp.transmit_timestamp','tcp.options',
           'tcp.payload','mqtt.msg','dns.qry.name','http.referer','mqtt.topic'}

numeric_cols = [col for col in peek.columns
                if col not in (LABEL_COL, TYPE_COL) and col not in ID_LIKE
                and pd.api.types.is_numeric_dtype(peek[col])]
keep = numeric_cols + [LABEL_COL, TYPE_COL, SRC_COL, DST_COL]
print(f'{len(numeric_cols)} candidate numeric features')
del peek; gc.collect()


In [ ]:
t0 = time.time()
total = sum(1 for _ in open(CSV_PATH, encoding='utf-8', errors='ignore')) - 1
n_blocks = max(1, SAMPLE_ROWS // BLOCK)
stride   = max(BLOCK, total // n_blocks)
print(f'{total:,} rows -> {n_blocks} contiguous blocks of {BLOCK:,}, stride {stride:,}')

parts, unparsable, taken = [], {c: 0 for c in numeric_cols}, 0
for b in range(n_blocks):
    start = b * stride
    if start >= total:
        break
    chunk = pd.read_csv(CSV_PATH, usecols=keep, skiprows=range(1, start + 1),
                        nrows=BLOCK, low_memory=False)
    for col in numeric_cols:
        conv = pd.to_numeric(chunk[col], errors='coerce')
        unparsable[col] += int(conv.isna().sum() - chunk[col].isna().sum())
        chunk[col] = conv.astype(np.float32)
    chunk[LABEL_COL] = (pd.to_numeric(chunk[LABEL_COL], errors='coerce')
                          .fillna(0).astype(np.int8))
    parts.append(chunk); taken += len(chunk)

df = pd.concat(parts, ignore_index=True); del parts; gc.collect()
print(f'loaded {len(df):,} flows in {time.time()-t0:.0f}s')


In [ ]:
n = len(df)
dead = [col for col, v in unparsable.items() if v > 0.01 * n]
if dead:
    print(f'dropping {len(dead)} text columns: {dead}')
    numeric_cols = [col for col in numeric_cols if col not in dead]

print(f'features: {len(numeric_cols)}')
print()
print(df[TYPE_COL].value_counts())

present = set(df[TYPE_COL].unique())
missing = [a for a in HELDOUT_ATTACKS if a not in present]
assert not missing, f'held-out families absent from the sample: {missing}'
print(f'\nheld-out families all present: {HELDOUT_ATTACKS}')


## 2 — Build the graph

Flows are linked when they share an endpoint **and** fall inside the same sliding
window of `GRAPH_WINDOW` consecutive rows. Row order stands in for time: the
capture is written in the order packets arrived.

Each flow also gets a self-loop, so it never loses its own signal when its
neighbours are averaged in.


In [ ]:
def build_flow_graph(src_host, dst_host, window=512, max_deg=8, seed=0):
    """Returns edge_index [2, E]; messages travel source -> target."""
    rng = np.random.default_rng(seed)
    n = len(src_host)
    rows, cols = [], []

    for start in range(0, n, window):
        blk = np.arange(start, min(start + window, n))
        if len(blk) < 2:
            continue
        buckets = {}
        for local, (a, b) in enumerate(zip(src_host[blk], dst_host[blk])):
            buckets.setdefault(('s', a), []).append(local)
            buckets.setdefault(('d', b), []).append(local)

        for members in buckets.values():
            if len(members) < 2:
                continue
            m = np.asarray(members)
            for i in m:
                others = m[m != i]
                if len(others) > max_deg:
                    others = rng.choice(others, max_deg, replace=False)
                rows.append(blk[others])
                cols.append(np.full(len(others), blk[i]))

    if not rows:
        raise RuntimeError('no edges built — check the endpoint columns')

    ei = np.vstack([np.concatenate(rows), np.concatenate(cols)])
    loops = np.arange(n)
    return np.hstack([ei, np.vstack([loops, loops])]).astype(np.int64)


In [ ]:
src_codes = pd.factorize(df[SRC_COL].astype(str))[0]
dst_codes = pd.factorize(df[DST_COL].astype(str))[0]
print(f'distinct source hosts {src_codes.max()+1} | destination hosts {dst_codes.max()+1}')

t0 = time.time()
edge_index = build_flow_graph(src_codes, dst_codes,
                              window=GRAPH_WINDOW, max_deg=MAX_DEGREE, seed=SEED)
print(f'graph built in {time.time()-t0:.0f}s')
print(f'  nodes {len(df):,}')
print(f'  edges {edge_index.shape[1]:,}  (avg degree {edge_index.shape[1]/len(df):.1f})')


> ⚠ If the host counts above are very small — a handful of addresses — the graph
> carries little structure and the GAT has little to work with. That is a property
> of a lab testbed, and worth saying out loud in the defence rather than hiding.


## 3 — Features and the zero-day split

Scaling is fitted on the training rows only. The held-out attack families appear
**nowhere** in training — not even as negatives.


In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
const = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=const)
print(f'dropped {len(const)} constant columns -> {X.shape[1]} features')

y     = df[LABEL_COL].values.astype(np.float32)
atype = df[TYPE_COL].values

is_heldout = np.isin(atype, HELDOUT_ATTACKS)
rng = np.random.default_rng(SEED)

# every flow of a held-out family goes to test; the rest split 70/30
rest = np.where(~is_heldout)[0]
rng.shuffle(rest)
cut = int(0.7 * len(rest))
train_idx = rest[:cut]
test_seen = rest[cut:]
test_zero = np.where(is_heldout)[0]

scaler = StandardScaler().fit(X.values[train_idx])
Xs = scaler.transform(X.values).astype(np.float32)

print(f'train              {len(train_idx):,}  (attacks {int(y[train_idx].sum()):,})')
print(f'test, seen types   {len(test_seen):,}  (attacks {int(y[test_seen].sum()):,})')
print(f'test, ZERO-DAY     {len(test_zero):,}  (all attacks)')

assert y[train_idx][np.isin(atype[train_idx], HELDOUT_ATTACKS)].size == 0, \
    'LEAK: a held-out family reached the training set'
print('\nno held-out family in training — split is clean')


## 4 — The two models

`GATLayer` is written out in full rather than imported, so every line is
inspectable and the only dependency is PyTorch itself.


In [ ]:
import torch.nn as nn, torch.nn.functional as Fn

def segment_softmax(scores, index, num_nodes):
    """Softmax across all edges sharing a target node."""
    mx = scores.new_full((num_nodes, scores.size(1)), float('-inf'))
    mx = mx.index_reduce(0, index, scores, reduce='amax', include_self=True)
    mx = torch.nan_to_num(mx, neginf=0.0)
    ex = torch.exp(scores - mx[index])
    den = torch.zeros_like(mx).index_add_(0, index, ex)
    return ex / (den[index] + 1e-16)

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, concat=True, dropout=0.2):
        super().__init__()
        self.heads, self.out_dim, self.concat, self.dropout = heads, out_dim, concat, dropout
        self.lin = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_dim))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_dim))
        self.bias = nn.Parameter(torch.zeros(heads * out_dim if concat else out_dim))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src); nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x, edge_index):
        N = x.size(0); src, dst = edge_index[0], edge_index[1]
        h = self.lin(x).view(N, self.heads, self.out_dim)
        e = Fn.leaky_relu((h * self.att_src).sum(-1)[src]
                          + (h * self.att_dst).sum(-1)[dst], 0.2)
        alpha = Fn.dropout(segment_softmax(e, dst, N), self.dropout, self.training)
        out = torch.zeros(N, self.heads, self.out_dim, device=x.device, dtype=x.dtype)
        out.index_add_(0, dst, h[src] * alpha.unsqueeze(-1))
        out = out.reshape(N, -1) if self.concat else out.mean(1)
        return out + self.bias

class GAT(nn.Module):
    def __init__(self, in_dim, hidden=32, heads=4, dropout=0.3):
        super().__init__()
        self.g1 = GATLayer(in_dim, hidden, heads, True, dropout)
        self.g2 = GATLayer(hidden * heads, hidden, 1, False, dropout)
        self.head = nn.Linear(hidden + in_dim, 1)   # raw features skip-connected
        self.dropout = dropout
    def forward(self, x, ei):
        h = Fn.elu(self.g1(x, ei))
        h = Fn.dropout(h, self.dropout, self.training)
        h = Fn.elu(self.g2(h, ei))
        return self.head(torch.cat([h, x], 1)).squeeze(-1)

class MLP(nn.Module):
    """The control: identical inputs, no graph."""
    def __init__(self, in_dim, hidden=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden*4), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(hidden*4, hidden), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1))
    def forward(self, x, ei=None):
        return self.net(x).squeeze(-1)


## 5 — Train both


In [ ]:
Xt  = torch.from_numpy(Xs)
yt  = torch.from_numpy(y)
eit = torch.from_numpy(edge_index)
tr  = torch.from_numpy(train_idx)

n_pos = max(int(y[train_idx].sum()), 1)
pos_weight = torch.tensor((len(train_idx) - n_pos) / n_pos, dtype=torch.float32)
print(f'class weight for the positive class: {pos_weight.item():.2f}')

def train(model, name):
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); opt.zero_grad()
        loss = lossf(model(Xt, eit)[tr], yt[tr])
        loss.backward(); opt.step()
        if (ep + 1) % 30 == 0:
            print(f'  {name} epoch {ep+1:>3}/{EPOCHS}  loss {loss.item():.4f}')
    print(f'  {name} trained in {(time.time()-t0)/60:.1f} min')
    return model

mlp = train(MLP(Xt.shape[1]), 'MLP')
gat = train(GAT(Xt.shape[1], HIDDEN, HEADS), 'GAT')


## 6 — Results

Two separate questions are scored:

- **seen types** — attack families the model trained on
- **zero-day** — families it has never encountered


In [ ]:
def scores(model):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(Xt, eit)).numpy()

def metrics(p, idx, thr=0.5):
    pred, true = (p[idx] > thr).astype(int), y[idx].astype(int)
    tp = int(((pred==1)&(true==1)).sum()); fn = int(((pred==0)&(true==1)).sum())
    fp = int(((pred==1)&(true==0)).sum()); tn = int(((pred==0)&(true==0)).sum())
    rec  = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
    return dict(recall=rec, precision=prec,
                f1=2*prec*rec/max(prec+rec,1e-9),
                fpr=fp/max(fp+tn,1), tp=tp, fn=fn, fp=fp, tn=tn)

p_mlp, p_gat = scores(mlp), scores(gat)

rows = []
for name, p in [('MLP (no graph)', p_mlp), ('GAT (with graph)', p_gat)]:
    s = metrics(p, test_seen)
    z = metrics(p, test_zero)
    rows.append({'Model': name,
                 'Seen recall %':     round(s['recall']*100, 2),
                 'Seen precision %':  round(s['precision']*100, 2),
                 'Seen F1 %':         round(s['f1']*100, 2),
                 'Seen FPR %':        round(s['fpr']*100, 2),
                 'Zero-day recall %': round(z['recall']*100, 2)})

res = pd.DataFrame(rows)
display(res)


In [ ]:
gap_f1  = res.loc[1,'Seen F1 %']         - res.loc[0,'Seen F1 %']
gap_zero= res.loc[1,'Zero-day recall %'] - res.loc[0,'Zero-day recall %']
print(f'Graph contribution, seen types : {gap_f1:+.2f} F1 points')
print(f'Graph contribution, zero-day   : {gap_zero:+.2f} recall points')
print()
if gap_f1 > 2 or gap_zero > 2:
    print('The graph adds measurable value over the identical no-graph model.')
else:
    print('The graph adds little here. That is a finding, not a failure —')
    print('report it honestly and say what it implies for the design.')


### Zero-day recall broken down by family


In [ ]:
rows = []
for fam in HELDOUT_ATTACKS:
    m = np.where(atype == fam)[0]
    if len(m) == 0:
        continue
    rows.append({'Family': fam, 'Samples': len(m),
                 'MLP %': round(float((p_mlp[m] > 0.5).mean()*100), 2),
                 'GAT %': round(float((p_gat[m] > 0.5).mean()*100), 2)})
per_fam = pd.DataFrame(rows)
display(per_fam)


In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

fig, ax = plt.subplots(figsize=(7.5, 4))
xpos = np.arange(len(per_fam)); w = 0.38
ax.bar(xpos - w/2, per_fam['MLP %'], w, label='MLP (no graph)', color='#8fa9c0')
ax.bar(xpos + w/2, per_fam['GAT %'], w, label='GAT (with graph)', color='#2a9d8f')
ax.set_xticks(xpos); ax.set_xticklabels(per_fam['Family'])
ax.set_ylabel('Recall (%)'); ax.set_ylim(0, 100)
ax.set_title('Zero-day families: does graph context help?')
ax.legend(); ax.grid(axis='y', alpha=.25)
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gnn_zeroday.png', dpi=150)
plt.show()


## 7 — Save


In [ ]:
res.to_csv(f'{OUT_DIR}/gnn_comparison.csv', index=False)
per_fam.to_csv(f'{OUT_DIR}/gnn_zeroday_by_family.csv', index=False)
torch.save(gat.state_dict(), f'{OUT_DIR}/gat.pt')

summary = {
    'flows'            : int(len(df)),
    'features'         : int(Xt.shape[1]),
    'edges'            : int(edge_index.shape[1]),
    'avg_degree'       : round(edge_index.shape[1]/len(df), 2),
    'distinct_src'     : int(src_codes.max()+1),
    'distinct_dst'     : int(dst_codes.max()+1),
    'heldout_families' : HELDOUT_ATTACKS,
    'train_flows'      : int(len(train_idx)),
    'graph_gain_f1'    : float(gap_f1),
    'graph_gain_zeroday': float(gap_zero),
}
with open(f'{OUT_DIR}/gnn_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('saved to', os.path.abspath(OUT_DIR))
print(json.dumps(summary, indent=2))


---

## What this result means

**If the GAT beats the MLP** — context carries information that per-flow features do
not. That is direct evidence for the correlation layer in the proposal, measured
rather than asserted.

**If the two are level** — on this dataset, in this testbed, neighbouring flows add
nothing the features did not already say. Report it plainly; a negative result that
is properly controlled is still a result, and it points at what the honeypot data
would need to provide instead.

Either way the MLP control is what makes the claim defensible. Without it, a higher
GAT number proves nothing: it could come from the extra parameters alone.

**Next (stage 2):** attach timestamps to edges so the model can tell *came before*
from *came after* — that is the step from GAT to Temporal GAT.
